## Events Order Audit

In [9]:
"""
audit_event_order.py
====================
Drop this file into notebooks/ and import it, or paste the cells directly
into a Jupyter notebook.

Usage
-----
    from audit_event_order import audit_event_order, audit_source_id_order, audit_both

    # Audit the numeric eventId field only
    result = audit_event_order("g2411061")

    # Audit the numeric source "id" field only
    result = audit_source_id_order("g2411061")

    # Audit BOTH fields side-by-side in a single report (recommended)
    result = audit_both("g2411061")

The functions look for the file in:
    data/raw/PRD/2025-2026/matches/

Adjust BASE_DIR below if your layout differs.
"""

from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
import pandas as pd

# ── Configuration ──────────────────────────────────────────────────────────────
# Adjust this to the root of your project (the folder that contains data/, src/, etc.)
PROJECT_ROOT = Path.cwd().parent   # notebooks/ → project root

BASE_DIR = PROJECT_ROOT / "data" / "raw" / "PRD" / "2025-2026" / "matches"


# ── JSONP parser (same strip strategy used across the pipeline) ────────────────
def _load_jsonp(path: Path) -> dict:
    raw = path.read_text(encoding="utf-8")
    start = raw.index("{")
    payload = raw[start:].rstrip()
    if payload.endswith(")"):
        payload = payload[:-1]
    return json.loads(payload)


# ── File resolver ──────────────────────────────────────────────────────────────
def _resolve_file(match_id: str, base_dir: Path) -> Path:
    stem = Path(match_id).stem
    candidates = list(base_dir.glob(f"{stem}*"))
    if not candidates:
        raise FileNotFoundError(
            f"No file matching '{stem}*' found in:\n  {base_dir}\n"
            f"Check that BASE_DIR and match_id are correct."
        )
    return candidates[0]


# ── Core audit logic (shared by both functions) ────────────────────────────────
def _audit_field(raw_events: list, field: str) -> dict:
    """
    Given the raw event list and a field name ("eventId" or "id"),
    compute sequentiality and JSON-order metrics.

    Returns a dict with:
        df           : full DataFrame (one row per event)
        sequential   : bool
        gaps         : list[int]
        duplicates   : list[int]
        order_ok     : bool
        out_of_order : DataFrame of misplaced events
        min_val      : int
        max_val      : int
    """
    records = []
    for json_pos, ev in enumerate(raw_events):
        records.append({
            "json_pos"  : json_pos,
            "value"     : ev.get(field),       # the field being audited
            "event_id"  : ev.get("eventId"),   # always include both for cross-ref
            "source_id" : ev.get("id"),
            "type_id"   : ev.get("typeId"),
            "period_id" : ev.get("periodId"),
            "time_min"  : ev.get("timeMin"),
            "time_sec"  : ev.get("timeSec"),
            "timestamp" : ev.get("timeStamp"),
        })

    df = pd.DataFrame(records)

    # Cast field values to int for numeric comparison
    values     = df["value"].dropna().astype(int).tolist()
    unique_vals = sorted(set(values))
    min_val, max_val = unique_vals[0], unique_vals[-1]

    # Sequentiality: check if IDs form a gapless range of length n
    expected_range = list(range(min_val, min_val + len(raw_events)))
    gaps       = sorted(set(expected_range) - set(values))
    counts     = Counter(values)
    duplicates = sorted(k for k, v in counts.items() if v > 1)
    sequential = (not gaps and not duplicates)

    # Order check: rank events by the audited field; compare to json_pos
    df["_rank"] = df["value"].rank(method="first").astype(int) - 1  # 0-based
    out_of_order = df[df["json_pos"] != df["_rank"]].copy()
    order_ok     = out_of_order.empty

    return {
        "df"          : df.drop(columns=["_rank"]),
        "sequential"  : sequential,
        "gaps"        : gaps,
        "duplicates"  : duplicates,
        "order_ok"    : order_ok,
        "out_of_order": out_of_order.drop(columns=["_rank"]),
        "min_val"     : min_val,
        "max_val"     : max_val,
    }


# ── Section printer ────────────────────────────────────────────────────────────
def _print_field_section(label: str, result: dict, n_events: int) -> None:
    """Print the audit block for one field."""
    sequential   = result["sequential"]
    gaps         = result["gaps"]
    duplicates   = result["duplicates"]
    order_ok     = result["order_ok"]
    out_of_order = result["out_of_order"]

    print(f"\n{'─' * 60}")
    print(f"  FIELD: {label}")
    print(f"{'─' * 60}")
    print(f"  Range       : {result['min_val']} → {result['max_val']}")
    print(f"  Step size   : {(result['max_val'] - result['min_val']) / max(n_events - 1, 1):.2f}  "
          f"(would be 1.00 if perfectly sequential)")

    print(f"\n  🔢  Sequential (no gaps/dupes) : {'✅ YES' if sequential else '❌ NO'}")
    if gaps:
        preview = gaps[:20]
        print(f"      Missing values ({len(gaps)} total) : {preview}{'...' if len(gaps) > 20 else ''}")
    if duplicates:
        print(f"      Duplicate values : {duplicates}")

    print(f"\n  🔀  JSON order == field order  : {'✅ YES' if order_ok else '❌ NO'}")
    if not order_ok:
        n = len(out_of_order)
        print(f"      {n} event(s) are out of position in the JSON array.")
        cols = ["json_pos", "value", "event_id", "source_id",
                "type_id", "period_id", "time_min", "time_sec", "timestamp"]
        # only show columns that exist
        cols = [c for c in cols if c in out_of_order.columns]
        print(f"      First 10 out-of-order rows:")
        print(out_of_order[cols].head(10).to_string(index=False))

        type_counts = (
            out_of_order["type_id"]
            .value_counts()
            .rename_axis("type_id")
            .reset_index(name="count")
        )
        print(f"\n      Out-of-order breakdown by typeId:")
        print(type_counts.to_string(index=False))

    # Interpretation
    print(f"\n  📋  Interpretation:")
    if sequential and order_ok:
        print(f"      ✅ Perfectly sequential AND already in JSON order.")
    elif sequential and not order_ok:
        print(f"      ⚠️  Values are sequential but JSON order is wrong.")
        print(f"         → Provider appended some events out of position.")
        print(f"         → Always sort by this field before processing.")
    elif not sequential and order_ok:
        print(f"      ⚠️  JSON order is correct but values are not sequential.")
        print(f"         → Gaps likely caused by deleted events (typeId=43)")
        print(f"         → or by the provider assigning non-consecutive IDs.")
    else:
        print(f"      ❌ Values are NOT sequential AND JSON order is wrong.")
        print(f"         → Sort by field before processing; check for deletions.")


# ── Public API ─────────────────────────────────────────────────────────────────

def audit_event_order(match_id: str, base_dir: Path | str = BASE_DIR) -> dict:
    """Audit the numeric `eventId` field only."""
    base_dir  = Path(base_dir)
    file_path = _resolve_file(match_id, base_dir)
    data       = _load_jsonp(file_path)
    raw_events = data.get("liveData", {}).get("event", [])
    if not raw_events:
        raise ValueError(f"No events found in liveData.event for {file_path.name}")

    result = _audit_field(raw_events, "eventId")

    sep = "=" * 60
    print(sep)
    print(f"  EVENT ORDER AUDIT  —  eventId")
    print(f"  File : {file_path.name}   ({len(raw_events)} events)")
    print(sep)
    _print_field_section("eventId  (provider sequence number)", result, len(raw_events))
    print(f"\n{sep}\n")

    return {
        "file_path"                   : file_path,
        "total_events"                : len(raw_events),
        "min_event_id"                : result["min_val"],
        "max_event_id"                : result["max_val"],
        "expected_sequential"         : result["sequential"],
        "gaps"                        : result["gaps"],
        "duplicates"                  : result["duplicates"],
        "json_order_matches_id_order" : result["order_ok"],
        "out_of_order_events"         : result["out_of_order"],
        "summary_df"                  : result["df"],
    }


def audit_source_id_order(match_id: str, base_dir: Path | str = BASE_DIR) -> dict:
    """Audit the numeric `id` field (source event identifier) only."""
    base_dir  = Path(base_dir)
    file_path = _resolve_file(match_id, base_dir)
    data       = _load_jsonp(file_path)
    raw_events = data.get("liveData", {}).get("event", [])
    if not raw_events:
        raise ValueError(f"No events found in liveData.event for {file_path.name}")

    result = _audit_field(raw_events, "id")

    sep = "=" * 60
    print(sep)
    print(f"  EVENT ORDER AUDIT  —  source id")
    print(f"  File : {file_path.name}   ({len(raw_events)} events)")
    print(sep)
    _print_field_section("id  (source numeric identifier)", result, len(raw_events))
    print(f"\n{sep}\n")

    return {
        "file_path"                      : file_path,
        "total_events"                   : len(raw_events),
        "min_source_id"                  : result["min_val"],
        "max_source_id"                  : result["max_val"],
        "expected_sequential"            : result["sequential"],
        "gaps"                           : result["gaps"],
        "duplicates"                     : result["duplicates"],
        "json_order_matches_id_order"    : result["order_ok"],
        "out_of_order_events"            : result["out_of_order"],
        "summary_df"                     : result["df"],
    }


def audit_both(match_id: str, base_dir: Path | str = BASE_DIR) -> dict:
    """
    Audit BOTH `eventId` and `id` fields in a single combined report.
    Also shows a cross-comparison: do the two fields agree on ordering?

    Returns a dict with keys  'event_id'  and  'source_id',
    each containing the same sub-dict as the individual audit functions.
    """
    base_dir  = Path(base_dir)
    file_path = _resolve_file(match_id, base_dir)
    data       = _load_jsonp(file_path)
    raw_events = data.get("liveData", {}).get("event", [])
    if not raw_events:
        raise ValueError(f"No events found in liveData.event for {file_path.name}")

    r_eid = _audit_field(raw_events, "eventId")
    r_sid = _audit_field(raw_events, "id")

    # Cross-comparison: do eventId rank and source id rank agree?
    df_eid = r_eid["df"].rename(columns={"value": "eventId_val"})
    df_sid = r_sid["df"].rename(columns={"value": "sourceId_val"})[["json_pos", "sourceId_val"]]
    merged = df_eid.merge(df_sid, on="json_pos")
    merged["eid_rank"] = merged["eventId_val"].rank(method="first").astype(int)
    merged["sid_rank"] = merged["sourceId_val"].rank(method="first").astype(int)
    rank_agreement     = (merged["eid_rank"] == merged["sid_rank"]).all()
    rank_disagreements = merged[merged["eid_rank"] != merged["sid_rank"]]

    # ── Print combined report ──────────────────────────────────────────────────
    sep = "=" * 60
    print(sep)
    print(f"  COMBINED EVENT ORDER AUDIT")
    print(f"  File : {file_path.name}   ({len(raw_events)} events)")
    print(sep)

    _print_field_section("eventId  (provider sequence number)", r_eid, len(raw_events))
    _print_field_section("id  (source numeric identifier)",     r_sid, len(raw_events))

    print(f"\n{'─' * 60}")
    print(f"  CROSS-COMPARISON: do eventId and id agree on order?")
    print(f"{'─' * 60}")
    print(f"  🔗  eventId rank == source id rank : {'✅ YES' if rank_agreement else '❌ NO'}")
    if not rank_agreement:
        n = len(rank_disagreements)
        print(f"      {n} event(s) where the two fields imply a different order.")
        print(f"      This means id and eventId were NOT assigned at the same time,")
        print(f"      or events were re-numbered after initial assignment.")
        cols = ["json_pos", "eventId_val", "sourceId_val", "type_id",
                "period_id", "time_min", "timestamp"]
        cols = [c for c in cols if c in rank_disagreements.columns]
        print(rank_disagreements[cols].head(10).to_string(index=False))

    print(f"\n{sep}\n")

    return {
        "event_id"  : {
            "min"          : r_eid["min_val"],
            "max"          : r_eid["max_val"],
            "sequential"   : r_eid["sequential"],
            "gaps"         : r_eid["gaps"],
            "duplicates"   : r_eid["duplicates"],
            "order_ok"     : r_eid["order_ok"],
            "out_of_order" : r_eid["out_of_order"],
            "summary_df"   : r_eid["df"],
        },
        "source_id" : {
            "min"          : r_sid["min_val"],
            "max"          : r_sid["max_val"],
            "sequential"   : r_sid["sequential"],
            "gaps"         : r_sid["gaps"],
            "duplicates"   : r_sid["duplicates"],
            "order_ok"     : r_sid["order_ok"],
            "out_of_order" : r_sid["out_of_order"],
            "summary_df"   : r_sid["df"],
        },
        "ranks_agree"         : rank_agreement,
        "rank_disagreements"  : rank_disagreements,
        "file_path"           : file_path,
        "total_events"        : len(raw_events),
    }


# ── Tiebreaker audit ───────────────────────────────────────────────────────────

def audit_tiebreaker(match_id: str, base_dir: Path | str = BASE_DIR) -> dict:
    """
    For every group of events that share the same (period, minute, second),
    check whether sorting by eventId or by source id produces the same order
    as their appearance in the raw JSON array.

    The JSON array order is treated as ground truth (it is the order the
    provider recorded the events).  A field is a "reliable tiebreaker" when,
    inside every tied group, its ascending order matches the JSON order.

    Parameters
    ----------
    match_id : str
        File stem or filename. Resolved against base_dir.
    base_dir : Path | str
        Folder containing raw match files.

    Returns
    -------
    dict with keys:
        file_path              : Path
        total_events           : int
        total_tied_groups      : int   – groups with > 1 event at same timestamp
        total_tied_events      : int   – events that belong to a tied group
        eventId_reliable       : bool  – True if eventId is a perfect tiebreaker
        source_id_reliable     : bool  – True if source id is a perfect tiebreaker
        eventId_violations     : pd.DataFrame  – tied groups where eventId order != JSON order
        source_id_violations   : pd.DataFrame  – tied groups where source id order != JSON order
        tied_groups_df         : pd.DataFrame  – all tied events with both fields and json_pos
    """
    base_dir  = Path(base_dir)
    file_path = _resolve_file(match_id, base_dir)
    data       = _load_jsonp(file_path)
    raw_events = data.get("liveData", {}).get("event", [])
    if not raw_events:
        raise ValueError(f"No events found in liveData.event for {file_path.name}")

    # ── Build flat DataFrame ───────────────────────────────────────────────────
    records = []
    for json_pos, ev in enumerate(raw_events):
        records.append({
            "json_pos"  : json_pos,
            "event_id"  : ev.get("eventId"),
            "source_id" : int(ev.get("id")) if ev.get("id") is not None else None,
            "type_id"   : ev.get("typeId"),
            "period_id" : ev.get("periodId"),
            "time_min"  : ev.get("timeMin"),
            "time_sec"  : ev.get("timeSec"),
            "timestamp" : ev.get("timeStamp"),
        })

    df = pd.DataFrame(records)
    df["event_id"]  = pd.to_numeric(df["event_id"],  errors="coerce")
    df["source_id"] = pd.to_numeric(df["source_id"], errors="coerce")

    # ── Isolate tied groups (same period + minute + second) ───────────────────
    group_key = ["period_id", "time_min", "time_sec"]
    group_sizes = df.groupby(group_key, dropna=False)["json_pos"].transform("count")
    tied_df = df[group_sizes > 1].copy()

    tied_groups = list(tied_df.groupby(group_key, dropna=False))
    total_tied_groups = len(tied_groups)
    total_tied_events = len(tied_df)

    # ── Check each field inside every tied group ───────────────────────────────
    def _check_tiebreaker(field: str) -> tuple[bool, pd.DataFrame]:
        """
        Returns (is_reliable, violations_df).
        is_reliable = True when every tied group is perfectly ordered by field.
        violations_df lists the groups + events where the order breaks down.
        """
        violation_rows = []

        for key, group in tied_groups:
            g = group.sort_values("json_pos").reset_index(drop=True)

            # Skip group if the field has any nulls
            if g[field].isna().any():
                # Nulls mean the field can't break ties — record as violation
                g["violation_reason"] = "null_value"
                g["group_key"] = str(key)
                violation_rows.append(g)
                continue

            field_order = g[field].tolist()
            is_sorted   = field_order == sorted(field_order)

            if not is_sorted:
                g["violation_reason"] = "wrong_order"
                g["group_key"] = str(key)
                violation_rows.append(g)

        if violation_rows:
            violations = pd.concat(violation_rows, ignore_index=True)
        else:
            violations = pd.DataFrame(columns=list(df.columns) + ["violation_reason", "group_key"])

        reliable = violations.empty
        return reliable, violations

    eid_reliable, eid_violations = _check_tiebreaker("event_id")
    sid_reliable, sid_violations = _check_tiebreaker("source_id")

    # ── Print report ───────────────────────────────────────────────────────────
    sep = "=" * 60
    print(sep)
    print(f"  TIEBREAKER AUDIT  —  same (period, minute, second)")
    print(f"  File : {file_path.name}   ({len(raw_events)} events)")
    print(sep)

    print(f"\n📦  Total events              : {len(df)}")
    print(f"    Tied groups               : {total_tied_groups}  "
          f"(groups with ≥ 2 events at the same timestamp)")
    print(f"    Events inside tied groups : {total_tied_events}")

    if total_tied_groups == 0:
        print("\n  ✅ No ties found — every (period, minute, second) is unique.")
        print(f"     Either field can serve as a tiebreaker trivially.\n")
        print(sep + "\n")
        return {
            "file_path"            : file_path,
            "total_events"         : len(df),
            "total_tied_groups"    : 0,
            "total_tied_events"    : 0,
            "eventId_reliable"     : True,
            "source_id_reliable"   : True,
            "eventId_violations"   : eid_violations,
            "source_id_violations" : sid_violations,
            "tied_groups_df"       : tied_df,
        }

    # ── eventId section ────────────────────────────────────────────────────────
    print(f"\n{'─' * 60}")
    print(f"  TIEBREAKER: eventId")
    print(f"{'─' * 60}")
    print(f"  Reliable tiebreaker : {'✅ YES' if eid_reliable else '❌ NO'}")
    if not eid_reliable:
        n_viol = eid_violations["group_key"].nunique()
        print(f"  Groups with wrong order : {n_viol} / {total_tied_groups}")
        print(f"  First offending groups:")
        cols = ["group_key", "json_pos", "event_id", "source_id",
                "type_id", "time_min", "time_sec", "violation_reason"]
        cols = [c for c in cols if c in eid_violations.columns]
        # show up to 3 groups
        top_groups = eid_violations["group_key"].unique()[:3]
        subset = eid_violations[eid_violations["group_key"].isin(top_groups)]
        print(subset[cols].to_string(index=False))

    # ── source id section ──────────────────────────────────────────────────────
    print(f"\n{'─' * 60}")
    print(f"  TIEBREAKER: source id  (numeric \"id\" field)")
    print(f"{'─' * 60}")
    print(f"  Reliable tiebreaker : {'✅ YES' if sid_reliable else '❌ NO'}")
    if not sid_reliable:
        n_viol = sid_violations["group_key"].nunique()
        print(f"  Groups with wrong order : {n_viol} / {total_tied_groups}")
        print(f"  First offending groups:")
        cols = ["group_key", "json_pos", "event_id", "source_id",
                "type_id", "time_min", "time_sec", "violation_reason"]
        cols = [c for c in cols if c in sid_violations.columns]
        top_groups = sid_violations["group_key"].unique()[:3]
        subset = sid_violations[sid_violations["group_key"].isin(top_groups)]
        print(subset[cols].to_string(index=False))

    # ── Summary verdict ────────────────────────────────────────────────────────
    print(f"\n{'─' * 60}")
    print(f"  VERDICT")
    print(f"{'─' * 60}")
    if eid_reliable and sid_reliable:
        print("  ✅ Both eventId and source id are reliable tiebreakers.")
        print("     Either can be used to sort within tied timestamps.")
    elif eid_reliable and not sid_reliable:
        print("  ✅ Use eventId as your tiebreaker.")
        print("  ❌ source id does NOT reliably preserve JSON order within ties.")
    elif not eid_reliable and sid_reliable:
        print("  ✅ Use source id as your tiebreaker.")
        print("  ❌ eventId does NOT reliably preserve JSON order within ties.")
    else:
        print("  ❌ Neither field reliably preserves JSON order within ties.")
        print("     The only safe ordering is: period → minute → second → json_pos.")
        print("     Consider keeping the original JSON position as an explicit column.")

    print(f"\n{sep}\n")

    return {
        "file_path"            : file_path,
        "total_events"         : len(df),
        "total_tied_groups"    : total_tied_groups,
        "total_tied_events"    : total_tied_events,
        "eventId_reliable"     : eid_reliable,
        "source_id_reliable"   : sid_reliable,
        "eventId_violations"   : eid_violations,
        "source_id_violations" : sid_violations,
        "tied_groups_df"       : tied_df,
    }

In [8]:
result = audit_tiebreaker("t6q5ftwcnm8os0cb3stk1kic")   # just the match_id / filename stem

  TIEBREAKER AUDIT  —  same (period, minute, second)
  File : t6q5ftwcnm8os0cb3stk1kic   (1628 events)

📦  Total events              : 1628
    Tied groups               : 284  (groups with ≥ 2 events at the same timestamp)
    Events inside tied groups : 611

────────────────────────────────────────────────────────────
  TIEBREAKER: eventId
────────────────────────────────────────────────────────────
  Reliable tiebreaker : ❌ NO
  Groups with wrong order : 108 / 284
  First offending groups:
 group_key  json_pos  event_id  source_id  type_id  time_min  time_sec violation_reason
(1, 1, 48)        35        22 2862032503        1         1        48      wrong_order
(1, 1, 48)        36        15 2862032511       74         1        48      wrong_order
(1, 2, 55)        54       470 2862068481       61         2        55      wrong_order
(1, 2, 55)        55        27 2862033297       43         2        55      wrong_order
(1, 3, 49)        73        41 2862033895       43         3  

## Sequence Analysis

In [10]:
"""
sequence_analysis.py — Diagnostic toolkit for possession sequence validation
═════════════════════════════════════════════════════════════════════════════
Run in a Jupyter notebook after Phase 7 (classify) to validate sequence
calculations, detect anomalies, and understand distributions.

Usage:
    import psycopg2
    import pandas as pd
    from sequence_analysis import SequenceAnalyzer

    conn = psycopg2.connect("your_dsn")
    sa = SequenceAnalyzer(conn)

    # Full diagnostic report
    sa.run_full_report()

    # Or individual checks
    sa.distribution_start_types()
    sa.distribution_end_types()
    sa.check_orphaned_events()
    sa.check_carry_ordering()
    ...
"""

import json
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd

# ──────────────────────────────────────────────────────────────────────
# Data loading
# ──────────────────────────────────────────────────────────────────────

_EVENTS_QUERY = """
    SELECT
        e.event_id, e.match_id, e.period, e.minute, e.second,
        e.provider_event_id,
        e.event_type, e.outcome, e.type_id,
        e.team_id, e.source_team_id,
        e.player_id, e.player_name,
        e.team_name, e.opposition_team_name, e.h_a,
        e.x, e.y, e.end_x, e.end_y, e.xt,
        e.sequence_id, e.sequence_start, e.sequence_end,
        e.sequence_event_number,
        e.raw_data
    FROM silver.events e
    WHERE e.match_id = ANY(%s)
    ORDER BY e.match_id, e.period, e.minute, e.second, e.event_id
"""

_ALL_CLASSIFIED_MATCHES = """
    SELECT DISTINCT match_id
    FROM silver.events
    WHERE sequence_id IS NOT NULL
    ORDER BY match_id
"""


class SequenceAnalyzer:
    """
    Diagnostic toolkit for validating possession sequence classifications.

    Parameters
    ----------
    conn : psycopg2 connection
    match_ids : list[int] | None
        Specific matches to analyze. If None, loads all classified matches.
    limit : int | None
        Cap on number of matches to load (useful for large datasets).
    """

    def __init__(self, conn, match_ids=None, limit=None):
        self.conn = conn

        if match_ids is None:
            with conn.cursor() as cur:
                sql = _ALL_CLASSIFIED_MATCHES
                if limit:
                    sql += f" LIMIT {int(limit)}"
                cur.execute(sql)
                match_ids = [r[0] for r in cur.fetchall()]

        self.match_ids = match_ids
        self.df = pd.read_sql(_EVENTS_QUERY, conn, params=(match_ids,))
        self._prepare()

        print(f"Loaded {len(self.df):,} events across {len(match_ids)} matches")
        print(f"  In sequences: {self.df['sequence_id'].notna().sum():,}")
        print(f"  Outside sequences: {self.df['sequence_id'].isna().sum():,}")
        print(f"  Unique sequences: {self.df['sequence_id'].dropna().nunique():,}")

    def _prepare(self):
        """Derived columns used by multiple checks."""
        self.in_seq = self.df[self.df['sequence_id'].notna()].copy()
        self.out_seq = self.df[self.df['sequence_id'].isna()].copy()
        self.starts = self.in_seq[self.in_seq['sequence_start'] == True].copy()
        self.ends = self.in_seq[self.in_seq['sequence_end'] == True].copy()

        # Sequence-level summary
        if not self.in_seq.empty:
            self.seq_summary = (
                self.in_seq.groupby('sequence_id')
                .agg(
                    match_id=('match_id', 'first'),
                    team_name=('team_name', 'first'),
                    h_a=('h_a', 'first'),
                    length=('sequence_event_number', 'max'),
                    start_minute=('minute', 'first'),
                    end_minute=('minute', 'last'),
                    n_teams=('source_team_id', 'nunique'),
                    has_start=('sequence_start', 'any'),
                    has_end=('sequence_end', 'any'),
                )
                .reset_index()
            )
        else:
            self.seq_summary = pd.DataFrame()

    # ══════════════════════════════════════════════════════════════════
    # 1. DISTRIBUTIONS
    # ══════════════════════════════════════════════════════════════════

    def distribution_start_types(self) -> pd.DataFrame:
        """Distribution of event types that start sequences."""
        dist = (
            self.starts['event_type']
            .value_counts()
            .reset_index()
        )
        dist.columns = ['event_type', 'count']
        dist['pct'] = (dist['count'] / dist['count'].sum() * 100).round(1)
        print("\n═══ Sequence START type distribution ═══")
        print(dist.to_string(index=False))
        return dist

    def distribution_end_types(self) -> pd.DataFrame:
        """Distribution of event types that end sequences."""
        dist = (
            self.ends['event_type']
            .value_counts()
            .reset_index()
        )
        dist.columns = ['event_type', 'count']
        dist['pct'] = (dist['count'] / dist['count'].sum() * 100).round(1)
        print("\n═══ Sequence END type distribution ═══")
        print(dist.to_string(index=False))
        return dist

    def distribution_sequence_lengths(self) -> pd.DataFrame:
        """Distribution of sequence lengths (number of events)."""
        if self.seq_summary.empty:
            print("No sequences to analyze.")
            return pd.DataFrame()

        stats = self.seq_summary['length'].describe()
        print("\n═══ Sequence length statistics ═══")
        print(stats.to_string())

        bins = [0, 1, 2, 3, 5, 10, 20, 50, 100, 999]
        labels = ['1', '2', '3', '4-5', '6-10', '11-20', '21-50', '51-100', '100+']
        self.seq_summary['length_bin'] = pd.cut(
            self.seq_summary['length'], bins=bins, labels=labels
        )
        dist = (
            self.seq_summary['length_bin']
            .value_counts()
            .sort_index()
            .reset_index()
        )
        dist.columns = ['length_range', 'count']
        dist['pct'] = (dist['count'] / dist['count'].sum() * 100).round(1)
        print("\n═══ Sequence length distribution ═══")
        print(dist.to_string(index=False))
        return dist

    def distribution_out_of_sequence(self) -> pd.DataFrame:
        """What event types are most commonly outside any sequence?"""
        dist = (
            self.out_seq['event_type']
            .value_counts()
            .reset_index()
        )
        dist.columns = ['event_type', 'count']
        dist['pct'] = (dist['count'] / dist['count'].sum() * 100).round(1)
        print("\n═══ Events OUTSIDE sequences (no sequence_id) ═══")
        print(dist.to_string(index=False))
        return dist

    # ══════════════════════════════════════════════════════════════════
    # 2. STRUCTURAL INTEGRITY CHECKS
    # ══════════════════════════════════════════════════════════════════

    def check_start_end_pairing(self) -> pd.DataFrame:
        """
        Every sequence should have exactly one start and one end.
        Returns sequences that violate this.
        """
        print("\n═══ Start/End pairing check ═══")

        no_start = self.seq_summary[~self.seq_summary['has_start']]
        no_end = self.seq_summary[~self.seq_summary['has_end']]

        print(f"  Sequences without a start event: {len(no_start)}")
        print(f"  Sequences without an end event:  {len(no_end)}")

        issues = pd.concat([
            no_start.assign(issue='missing_start'),
            no_end.assign(issue='missing_end'),
        ])

        if not issues.empty:
            print("\n  ⚠ Problematic sequences:")
            print(issues[['sequence_id', 'match_id', 'team_name', 'length', 'issue']]
                  .head(20).to_string(index=False))
        else:
            print("  ✓ All sequences have both start and end events.")

        return issues

    def check_meta_events_in_sequences(self) -> pd.DataFrame:
        """
        Meta events (Start, End, SubstitutionOn/Off) should NEVER
        appear inside a sequence.
        """
        META = {'Start', 'End', 'SubstitutionOn', 'SubstitutionOff',
                'FormationChange', 'FormationSet'}

        violations = self.in_seq[self.in_seq['event_type'].isin(META)]

        print("\n═══ Meta events inside sequences ═══")
        print(f"  Violations found: {len(violations)}")

        if not violations.empty:
            dist = violations['event_type'].value_counts()
            print("  Breakdown:")
            for evt, cnt in dist.items():
                print(f"    {evt}: {cnt}")
            print("\n  Sample violations:")
            print(violations[['event_id', 'match_id', 'event_type', 'sequence_id']]
                  .head(10).to_string(index=False))
        else:
            print("  ✓ No meta events found inside sequences.")

        return violations

    def check_excluded_periods(self) -> pd.DataFrame:
        """Events from period 14/16 should never be in a sequence."""
        violations = self.in_seq[self.in_seq['period'].isin([14, 16])]

        print("\n═══ Excluded period events in sequences ═══")
        print(f"  Period 14/16 events in sequences: {len(violations)}")

        if not violations.empty:
            print(violations[['event_id', 'match_id', 'period', 'event_type', 'sequence_id']]
                  .head(10).to_string(index=False))
        else:
            print("  ✓ No excluded-period events in sequences.")

        return violations

    # ══════════════════════════════════════════════════════════════════
    # 3. EDGE CASE VALIDATION
    # ══════════════════════════════════════════════════════════════════

    def check_carry_ordering(self) -> pd.DataFrame:
        """
        Carries should always be surrounded by events from the same team.
        If a carry is the first or last event of a sequence, or its
        neighbours are from a different team, the sort order may be wrong.
        """
        print("\n═══ Carry ordering check ═══")

        carries = self.in_seq[self.in_seq['event_type'] == 'Carry'].copy()
        if carries.empty:
            print("  No carries found in sequences.")
            return pd.DataFrame()

        issues = []
        for _, carry in carries.iterrows():
            seq_events = self.in_seq[
                self.in_seq['sequence_id'] == carry['sequence_id']
            ].sort_values('sequence_event_number')

            evt_num = carry['sequence_event_number']

            # Get previous and next events in the sequence
            prev_events = seq_events[seq_events['sequence_event_number'] == evt_num - 1]
            next_events = seq_events[seq_events['sequence_event_number'] == evt_num + 1]

            issue = None
            if prev_events.empty:
                issue = 'carry_is_first_in_sequence'
            elif next_events.empty:
                issue = 'carry_is_last_in_sequence'
            else:
                prev_team = prev_events.iloc[0]['source_team_id']
                carry_team = carry['source_team_id']
                if prev_team != carry_team:
                    issue = 'carry_different_team_from_prev'

            if issue:
                issues.append({
                    'event_id': carry['event_id'],
                    'match_id': carry['match_id'],
                    'sequence_id': carry['sequence_id'],
                    'sequence_event_number': evt_num,
                    'issue': issue,
                })

        issues_df = pd.DataFrame(issues)
        print(f"  Total carries in sequences: {len(carries)}")
        print(f"  Ordering anomalies: {len(issues_df)}")

        if not issues_df.empty:
            dist = issues_df['issue'].value_counts()
            for iss, cnt in dist.items():
                print(f"    {iss}: {cnt}")
            print("\n  Samples:")
            print(issues_df.head(10).to_string(index=False))

            # Show full context for first anomaly
            if len(issues_df) > 0:
                sample_seq = issues_df.iloc[0]['sequence_id']
                print(f"\n  Full context for sequence {sample_seq}:")
                ctx = self.in_seq[self.in_seq['sequence_id'] == sample_seq][
                    ['event_id', 'event_type', 'team_name', 'player_name',
                     'outcome', 'sequence_event_number']
                ]
                print(ctx.to_string(index=False))
        else:
            print("  ✓ All carries are correctly ordered within their sequences.")

        return issues_df

    def check_sandwich_exemptions(self) -> pd.DataFrame:
        """
        Find BallTouch/Error events from the opposing team that are
        INSIDE a sequence. These should only happen when the sandwich
        exemption is active (prev and next are same team).
        Flags cases where the sandwich condition doesn't hold.
        """
        print("\n═══ Sandwich exemption validation ═══")

        sandwich_types = {'BallTouch', 'Error'}
        candidates = self.in_seq[
            self.in_seq['event_type'].isin(sandwich_types)
        ].copy()

        if candidates.empty:
            print("  No BallTouch/Error events found in sequences.")
            return pd.DataFrame()

        issues = []
        valid_sandwiches = 0

        for _, row in candidates.iterrows():
            seq_events = self.in_seq[
                self.in_seq['sequence_id'] == row['sequence_id']
            ].sort_values('sequence_event_number')

            evt_num = row['sequence_event_number']
            prev_evts = seq_events[seq_events['sequence_event_number'] == evt_num - 1]
            next_evts = seq_events[seq_events['sequence_event_number'] == evt_num + 1]

            if prev_evts.empty or next_evts.empty:
                continue

            prev_team = prev_evts.iloc[0]['source_team_id']
            next_team = next_evts.iloc[0]['source_team_id']
            row_team = row['source_team_id']

            # The event is from a different team than prev/next
            if row_team != prev_team:
                if prev_team == next_team:
                    valid_sandwiches += 1
                else:
                    issues.append({
                        'event_id': row['event_id'],
                        'match_id': row['match_id'],
                        'sequence_id': row['sequence_id'],
                        'event_type': row['event_type'],
                        'evt_team': row['team_name'],
                        'prev_team': prev_evts.iloc[0]['team_name'],
                        'next_team': next_evts.iloc[0]['team_name'],
                        'issue': 'broken_sandwich',
                    })

        issues_df = pd.DataFrame(issues)
        print(f"  Cross-team BallTouch/Error events in sequences: {len(candidates)}")
        print(f"  Valid sandwiches (prev_team == next_team != evt_team): {valid_sandwiches}")
        print(f"  Broken sandwiches: {len(issues_df)}")

        if not issues_df.empty:
            print("\n  ⚠ Broken sandwich cases:")
            print(issues_df.head(10).to_string(index=False))
        else:
            print("  ✓ All cross-team BallTouch/Error events are valid sandwiches.")

        return issues_df

    def check_corner_awarded_handling(self) -> pd.DataFrame:
        """
        CornerAwarded should always be the LAST event of a sequence
        (sequence_end = True). It should never start a sequence.
        """
        print("\n═══ CornerAwarded handling ═══")

        corners_in = self.in_seq[self.in_seq['event_type'] == 'CornerAwarded']
        corners_out = self.out_seq[self.out_seq['event_type'] == 'CornerAwarded']

        print(f"  CornerAwarded in sequences:   {len(corners_in)}")
        print(f"  CornerAwarded outside sequences: {len(corners_out)}")

        # Should always end the sequence
        corners_not_end = corners_in[corners_in['sequence_end'] != True]
        print(f"  CornerAwarded NOT marked as end: {len(corners_not_end)}")

        # Should never start a sequence
        corners_start = corners_in[corners_in['sequence_start'] == True]
        print(f"  CornerAwarded marked as start: {len(corners_start)}")

        issues = pd.concat([
            corners_not_end.assign(issue='corner_not_end'),
            corners_start.assign(issue='corner_is_start'),
        ])

        if not issues.empty:
            print("\n  ⚠ Issues:")
            print(issues[['event_id', 'match_id', 'sequence_id', 'issue']]
                  .head(10).to_string(index=False))
        else:
            print("  ✓ All CornerAwarded events correctly end their sequences.")

        return issues

    def check_set_piece_starts(self) -> pd.DataFrame:
        """
        Passes with set-piece qualifiers (Q5, Q6, Q107) should always
        start a sequence. Checks for any that don't.
        """
        print("\n═══ Set-piece pass start check ═══")

        SET_PIECE_QIDS = {5, 6, 107}

        passes = self.df[self.df['event_type'] == 'Pass'].copy()

        def _has_set_piece_q(raw_data):
            if raw_data is None or (isinstance(raw_data, float) and np.isnan(raw_data)):
                return False
            if isinstance(raw_data, str):
                try:
                    raw_data = json.loads(raw_data)
                except:
                    return False
            if not isinstance(raw_data, dict):
                return False
            for q in raw_data.get('qualifier', []):
                if isinstance(q, dict):
                    qid = q.get('qualifierId', q.get('id'))
                    if qid is not None and int(qid) in SET_PIECE_QIDS:
                        return True
            return False

        passes['is_set_piece'] = passes['raw_data'].apply(_has_set_piece_q)
        sp_passes = passes[passes['is_set_piece']]

        sp_not_start = sp_passes[sp_passes['sequence_start'] != True]
        sp_not_in_seq = sp_passes[sp_passes['sequence_id'].isna()]

        print(f"  Total set-piece passes found: {len(sp_passes)}")
        print(f"  Set-piece passes that start a sequence: {(sp_passes['sequence_start'] == True).sum()}")
        print(f"  Set-piece passes NOT starting a sequence: {len(sp_not_start)}")
        print(f"  Set-piece passes outside any sequence: {len(sp_not_in_seq)}")

        if not sp_not_start.empty:
            # Breakdown by qualifier
            def _get_sp_type(raw_data):
                if raw_data is None or not isinstance(raw_data, (str, dict)):
                    return 'unknown'
                if isinstance(raw_data, str):
                    try:
                        raw_data = json.loads(raw_data)
                    except:
                        return 'unknown'
                for q in raw_data.get('qualifier', []):
                    qid = int(q.get('qualifierId', q.get('id', 0)))
                    if qid == 5: return 'free_kick'
                    if qid == 6: return 'corner'
                    if qid == 107: return 'throw_in'
                return 'unknown'

            sp_not_start['sp_type'] = sp_not_start['raw_data'].apply(_get_sp_type)
            print("\n  ⚠ Non-starting set-piece passes by type:")
            print(sp_not_start['sp_type'].value_counts().to_string())
            print("\n  Samples:")
            print(sp_not_start[['event_id', 'match_id', 'event_type', 'outcome',
                                 'sequence_id', 'sequence_start', 'sp_type']]
                  .head(10).to_string(index=False))
        else:
            print("  ✓ All set-piece passes correctly start sequences.")

        return sp_not_start

    def check_goals(self) -> pd.DataFrame:
        """
        Goals should always END a sequence.
        Goals should only START a sequence if they have Q9 (penalty)
        or Q5 (free kick).
        """
        print("\n═══ Goal handling ═══")

        DEAD_BALL_QIDS = {5, 9}

        goals = self.in_seq[self.in_seq['event_type'] == 'Goal'].copy()
        goals_out = self.out_seq[self.out_seq['event_type'] == 'Goal']

        print(f"  Goals in sequences: {len(goals)}")
        print(f"  Goals outside sequences: {len(goals_out)}")

        # All goals should end
        goals_not_end = goals[goals['sequence_end'] != True]
        print(f"  Goals NOT ending a sequence: {len(goals_not_end)}")

        # Goals that start — check if they have dead-ball qualifiers
        goals_that_start = goals[goals['sequence_start'] == True].copy()
        print(f"  Goals that START a sequence: {len(goals_that_start)}")

        def _has_dead_ball_q(raw_data):
            if raw_data is None or (isinstance(raw_data, float) and np.isnan(raw_data)):
                return False
            if isinstance(raw_data, str):
                try:
                    raw_data = json.loads(raw_data)
                except:
                    return False
            if not isinstance(raw_data, dict):
                return False
            for q in raw_data.get('qualifier', []):
                qid = int(q.get('qualifierId', q.get('id', 0)))
                if qid in DEAD_BALL_QIDS:
                    return True
            return False

        if not goals_that_start.empty:
            goals_that_start['is_dead_ball'] = goals_that_start['raw_data'].apply(
                _has_dead_ball_q
            )
            open_play_starts = goals_that_start[~goals_that_start['is_dead_ball']]
            print(f"  Goals starting with dead-ball qualifier: "
                  f"{goals_that_start['is_dead_ball'].sum()}")
            print(f"  ⚠ Open-play goals incorrectly starting: {len(open_play_starts)}")

            if not open_play_starts.empty:
                print(open_play_starts[
                    ['event_id', 'match_id', 'team_name', 'player_name', 'sequence_id']
                ].head(10).to_string(index=False))

        issues = pd.concat([
            goals_not_end.assign(issue='goal_not_ending'),
            goals_out.assign(issue='goal_outside_sequence'),
        ])
        return issues

    def check_single_event_sequences(self) -> pd.DataFrame:
        """
        Single-event sequences are valid (e.g. a clearance that
        immediately loses possession). But an unusually high proportion
        may indicate a classifier issue.
        """
        print("\n═══ Single-event sequences ═══")

        singles = self.seq_summary[self.seq_summary['length'] == 1]
        total = len(self.seq_summary)

        print(f"  Single-event sequences: {len(singles)} / {total} "
              f"({len(singles)/max(total,1)*100:.1f}%)")

        if not singles.empty:
            # What events create single-event sequences?
            single_ids = set(singles['sequence_id'])
            single_events = self.in_seq[self.in_seq['sequence_id'].isin(single_ids)]
            dist = single_events['event_type'].value_counts().reset_index()
            dist.columns = ['event_type', 'count']
            print("\n  Event types in single-event sequences:")
            print(dist.to_string(index=False))

        return singles

    def check_cross_team_events(self) -> pd.DataFrame:
        """
        A sequence is owned by one team, but can contain events from the
        other team (Challenge, Aerial, sandwich BallTouch/Error).
        Flags sequences with unexpectedly high cross-team event ratios.
        """
        print("\n═══ Cross-team events in sequences ═══")

        if self.in_seq.empty:
            print("  No sequences to analyze.")
            return pd.DataFrame()

        # Get the owning team per sequence (from the start event)
        seq_owners = (
            self.starts[['sequence_id', 'source_team_id']]
            .rename(columns={'source_team_id': 'owner_team'})
        )

        merged = self.in_seq.merge(seq_owners, on='sequence_id', how='left')
        merged['is_cross_team'] = merged['source_team_id'] != merged['owner_team']

        cross = merged[merged['is_cross_team']]
        print(f"  Total cross-team events in sequences: {len(cross)}")

        if not cross.empty:
            dist = cross['event_type'].value_counts().reset_index()
            dist.columns = ['event_type', 'count']
            print("\n  Cross-team event type distribution:")
            print(dist.to_string(index=False))

            # Flag sequences where > 50% of events are cross-team
            ct_ratio = (
                merged.groupby('sequence_id')
                .agg(
                    total=('is_cross_team', 'count'),
                    cross=('is_cross_team', 'sum'),
                )
            )
            ct_ratio['ratio'] = ct_ratio['cross'] / ct_ratio['total']
            high_cross = ct_ratio[ct_ratio['ratio'] > 0.5].reset_index()

            print(f"\n  Sequences with >50% cross-team events: {len(high_cross)}")
            if not high_cross.empty:
                print(high_cross.head(10).to_string(index=False))

                # Show context for worst case
                worst = high_cross.sort_values('ratio', ascending=False).iloc[0]
                print(f"\n  Full context for {worst['sequence_id']}:")
                ctx = self.in_seq[
                    self.in_seq['sequence_id'] == worst['sequence_id']
                ][['event_type', 'team_name', 'player_name', 'outcome',
                   'sequence_event_number']]
                print(ctx.to_string(index=False))

        return cross

    # ══════════════════════════════════════════════════════════════════
    # 4. SEQUENCE CONTEXT INSPECTOR
    # ══════════════════════════════════════════════════════════════════

    def inspect_sequence(self, sequence_id: str) -> pd.DataFrame:
        """Print full event context for a specific sequence, including
        2 events before and after for boundary context."""
        seq_events = self.df[self.df['sequence_id'] == sequence_id]
        if seq_events.empty:
            print(f"Sequence '{sequence_id}' not found.")
            return pd.DataFrame()

        match_id = seq_events.iloc[0]['match_id']
        match_df = self.df[self.df['match_id'] == match_id].copy()
        match_df = match_df.sort_values('event_id')

        # Find the range
        first_idx = match_df.index[match_df['event_id'] == seq_events.iloc[0]['event_id']][0]
        last_idx = match_df.index[match_df['event_id'] == seq_events.iloc[-1]['event_id']][0]
        pos_first = match_df.index.get_loc(first_idx)
        pos_last = match_df.index.get_loc(last_idx)

        start = max(0, pos_first - 2)
        end = min(len(match_df), pos_last + 3)

        context = match_df.iloc[start:end].copy()
        context['in_this_seq'] = context['sequence_id'] == sequence_id

        cols = ['event_id', 'minute', 'second', 'team_name', 'player_name',
                'event_type', 'outcome', 'sequence_id',
                'sequence_start', 'sequence_end', 'sequence_event_number',
                'in_this_seq']

        available = [c for c in cols if c in context.columns]

        print(f"\n═══ Sequence: {sequence_id} ═══")
        print(f"  Match: {match_id} | Team: {seq_events.iloc[0]['team_name']} | "
              f"Length: {len(seq_events)} events")
        print(f"  Context (± 2 events):\n")
        print(context[available].to_string(index=False))
        return context

    def inspect_match_sequences(self, match_id: int, minute_from: int = 0,
                                 minute_to: int = 999) -> pd.DataFrame:
        """Print all events for a match in a time window, with sequence info."""
        match_df = self.df[
            (self.df['match_id'] == match_id)
            & (self.df['minute'] >= minute_from)
            & (self.df['minute'] <= minute_to)
        ].sort_values('event_id')

        cols = ['event_id', 'minute', 'second', 'team_name', 'player_name',
                'event_type', 'outcome', 'sequence_id',
                'sequence_start', 'sequence_end', 'sequence_event_number']
        available = [c for c in cols if c in match_df.columns]

        print(f"\n═══ Match {match_id} — minutes {minute_from}-{minute_to} ═══")
        print(match_df[available].to_string(index=False))
        return match_df

    # ══════════════════════════════════════════════════════════════════
    # 5. PER-MATCH SUMMARY
    # ══════════════════════════════════════════════════════════════════

    def match_summary(self) -> pd.DataFrame:
        """Per-match breakdown: sequences per team, coverage ratio."""
        if self.in_seq.empty:
            print("No sequences.")
            return pd.DataFrame()

        per_match = (
            self.df.groupby('match_id')
            .agg(
                total_events=('event_id', 'count'),
                in_sequence=('sequence_id', lambda s: s.notna().sum()),
                n_sequences=('sequence_id', lambda s: s.dropna().nunique()),
            )
            .reset_index()
        )
        per_match['coverage_pct'] = (
            per_match['in_sequence'] / per_match['total_events'] * 100
        ).round(1)

        # Per team
        per_team = (
            self.in_seq.groupby(['match_id', 'h_a'])
            .agg(n_seq=('sequence_id', 'nunique'))
            .reset_index()
            .pivot(index='match_id', columns='h_a', values='n_seq')
            .rename(columns={'home': 'home_sequences', 'away': 'away_sequences'})
            .reset_index()
        )

        result = per_match.merge(per_team, on='match_id', how='left')

        print("\n═══ Per-match summary ═══")
        print(f"  Mean coverage: {result['coverage_pct'].mean():.1f}%")
        print(f"  Mean sequences per match: {result['n_sequences'].mean():.0f}")
        print(f"  Mean home sequences: {result['home_sequences'].mean():.0f}")
        print(f"  Mean away sequences: {result['away_sequences'].mean():.0f}")

        # Flag low-coverage matches
        low = result[result['coverage_pct'] < 50]
        if not low.empty:
            print(f"\n  ⚠ Low coverage matches (<50%): {len(low)}")
            print(low.to_string(index=False))

        return result

    # ══════════════════════════════════════════════════════════════════
    # 6. FULL REPORT
    # ══════════════════════════════════════════════════════════════════

    def run_full_report(self) -> Dict[str, pd.DataFrame]:
        """Run all checks and distributions. Returns dict of results."""
        print("╔════════════════════════════════════════════════════╗")
        print("║  POSSESSION SEQUENCE — DIAGNOSTIC REPORT          ║")
        print(f"║  Matches: {len(self.match_ids):<6}  "
              f"Events: {len(self.df):<10}            ║")
        print(f"║  Sequences: {self.seq_summary['sequence_id'].nunique() if not self.seq_summary.empty else 0:<6}"
              f"                                  ║")
        print("╚════════════════════════════════════════════════════╝")

        results = {}

        # Distributions
        results['start_types'] = self.distribution_start_types()
        results['end_types'] = self.distribution_end_types()
        results['lengths'] = self.distribution_sequence_lengths()
        results['out_of_sequence'] = self.distribution_out_of_sequence()

        # Structural checks
        results['pairing'] = self.check_start_end_pairing()
        results['meta_in_seq'] = self.check_meta_events_in_sequences()
        results['excluded_periods'] = self.check_excluded_periods()

        # Edge case checks
        results['carry_order'] = self.check_carry_ordering()
        results['sandwiches'] = self.check_sandwich_exemptions()
        results['corners'] = self.check_corner_awarded_handling()
        results['set_pieces'] = self.check_set_piece_starts()
        results['goals'] = self.check_goals()
        results['singles'] = self.check_single_event_sequences()
        results['cross_team'] = self.check_cross_team_events()

        # Summary
        results['match_summary'] = self.match_summary()

        # Final score
        print("\n╔════════════════════════════════════════════════════╗")
        print("║  SUMMARY                                           ║")
        print("╚════════════════════════════════════════════════════╝")

        checks = {
            'Start/end pairing':    len(results['pairing']) == 0,
            'No meta in sequences': len(results['meta_in_seq']) == 0,
            'No excluded periods':  len(results['excluded_periods']) == 0,
            'Carry ordering':       len(results['carry_order']) == 0,
            'Sandwich exemptions':  len(results['sandwiches']) == 0,
            'CornerAwarded':        len(results['corners']) == 0,
            'Set-piece starts':     len(results['set_pieces']) == 0,
        }

        for name, passed in checks.items():
            icon = '✓' if passed else '⚠'
            print(f"  {icon} {name}")

        n_pass = sum(checks.values())
        print(f"\n  {n_pass}/{len(checks)} checks passed.")

        return results

In [11]:
import psycopg2
from dotenv import load_dotenv
import os


load_dotenv()
dsn = os.environ.get("FOOTBALL_DB_DSN")

conn = psycopg2.connect(dsn)

# Full report — all checks at once
sa = SequenceAnalyzer(conn)              # loads all classified matches
results = sa.run_full_report()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_25892\2457835259.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  self.df = pd.read_sql(_EVENTS_QUERY, conn, params=(match_ids,))


Loaded 539,677 events across 257 matches
  In sequences: 391,585
  Outside sequences: 148,092
  Unique sequences: 58,920
╔════════════════════════════════════════════════════╗
║  POSSESSION SEQUENCE — DIAGNOSTIC REPORT          ║
║  Matches: 257     Events: 539677                ║
║  Sequences: 58920                                   ║
╚════════════════════════════════════════════════════╝

═══ Sequence START type distribution ═══
               event_type  count  pct
                     Pass  37750 64.1
                Clearance  10902 18.5
                   Tackle   5050  8.6
             Interception   4058  6.9
         Formation change    688  1.2
                    Claim    372  0.6
                     Goal     71  0.1
        Referee Drop Ball     26  0.0
                    Carry      1  0.0
             Offside Pass      1  0.0
Player becomes goalkeeper      1  0.0

═══ Sequence END type distribution ═══
       event_type  count  pct
             Pass  33422 56.7
        C

KeyboardInterrupt: 